### *This notebook serves as the master Pokemon data notebook. Other notebooks that are more narrowly focused on specific Pokemon data will run this notebook and use its functionality to simplify code writing and streamline functions like joining and webscraping.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
# Maximize display of all dataframes
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('future.no_silent_downcasting', True)

### Constants

In [ ]:
forms = ['Mega','Black','White','Alolan','Galarian','Hisuian','Paldean','Primal','Heat','Wash','Frost','Fan','Mow']

# Add new gens as necessary
rename_alternate_forms = {
    'GIRATINA ALTERED FORME':'GIRATINA',
    'GIRATINA ORIGIN FORME':'GIRATINA DISTORTION FORME',
    'DARMANITAN STANDARD MODE':'DARMANITAN',
    'TORNADUS INCARNATE FORME':'TORNADUS',
    'THUNDURUS INCARNATE FORME':'THUNDURUS',
    'LANDORUS INCARNATE FORME':'LANDORUS',
    'MELOETTA ARIA FORME':'MELOETTA',
    'KELDEO ORDINARY FORM':'KELDEO'
}

# Add new gens as necessary
redundant_pokemon = ['BASCULIN BLUE-STRIPED FORM','BASCULIN WHITE-STRIPED FORM']
alternate_forms = {
    'Mega':6, 
    'Galarian':8, 
    'Hisuian':8, 
    'Alolan':7, 
    'Paldean':9, 
    'Primal':6, 
    'Partner':'remove', 
    'Breed':9, 
    'Origin':8
    }

# Add new gens 
regional_pokedex_urls = {
    'Johto':'https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_Johto_Pok%C3%A9dex_number',
    'Hoenn':'https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_Hoenn_Pok%C3%A9dex_number_in_Generation_III',
    'Sinnoh':'https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_Sinnoh_Pok%C3%A9dex_number',
    'Unova (Black/White)':'https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_Unova_Pok%C3%A9dex_number_in_Pok%C3%A9mon_Black_and_White',
    'Unova (Black2/White2)':'https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_Unova_Pok%C3%A9dex_number_in_Pok%C3%A9mon_Black_2_and_White_2',
}

# Add new gens
regional_index_cap = {
    'Johto':261,
    'Hoenn':209,
    'Sinnoh':226,
    'Unova (Black/White)':164,
    'Unova (Black2/White2)':318
}

### Web Scraping Function

In [ ]:
# Scrape data from a web server
def scrape_table(url):
    
    # Exception handling for connection errors
    try:
        response = requests.get(url)
        html = response.text

    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error occurred: {e}")

    except requests.exceptions.ConnectionError as e:
        print(f"Connection Error occurred: {e}.")

    except Exception as e:
        print(f"An unexpected error occurred: {e}")

    soup = BeautifulSoup(html, 'html.parser')
    rows = soup.find_all('tr') # Retrieve all rows within the database
    
    data = []

    for row in rows:
        cols = row.find_all(['td','th']) # Retrieve all columns within each row
        row_data = []
        
        # Account for columns that may contain an images as values
        for col in cols:
            img = col.find('img')  # Check for an image inside each cell
            if img and not col.text.strip():
                img_url = f'https://www.serebii.net{img['src']}' # Get full image url 
                row_data.append(img_url) 
            else:
                row_data.append(col.text.strip())  # Otherwise, get text
        data.append(row_data)
    
    global df
    df = pd.DataFrame(data)
    return df

### Join Function

In [ ]:
# Add new columns to an existing dataframe 
def join(df_main, df_join, kind, key):
    valid_kinds = ['left', 'right', 'inner', 'outer']
    if kind not in valid_kinds:
        raise ValueError(f'Kind must be of {valid_kinds}')
    df_main = pd.merge(df_main, df_join, how = kind, on = key)
    return df_main

### Pass National Pokedex URL into Scraping Function and Prepare Dataframe for Cleaning

In [ ]:
pokedex_url = 'https://pokemondb.net/pokedex/all'
scrape_table(pokedex_url)
pokedex_df = df

# Realign dataframe
pokedex_df.columns = pokedex_df.iloc[0] 
pokedex_df = pokedex_df.iloc[1:]

### Apply Transformations to National Pokedex

In [ ]:
pokedex_df = pokedex_df.rename(columns = {'Total':'Base Stats', 'Name':'Pokémon'})
pokedex_df['Type'] = pokedex_df['Type'].str.replace(' ','|')
pokedex_df['#'] = pokedex_df['#'].astype('int')

# Clean up alternative form strings
for form in forms:
    mask = pokedex_df['Pokémon'].str.contains(fr'\b{form}\b', case=False, na=False)
    pokedex_df.loc[mask, 'Pokémon'] = (
        pokedex_df.loc[mask, 'Pokémon'].str.replace(fr'^.*?\b{form}\b\s*', f'{form} ', regex=True, case=False)
    )

# Rename alternate form names for easier pokemon stats widget querying
pokedex_df['Pokémon'] = pokedex_df['Pokémon'].str.upper().str.strip().replace(rename_alternate_forms)

# Remove redundant pokemon namings
pokedex_df = pokedex_df[~pokedex_df['Pokémon'].isin(redundant_pokemon)]

# Filter alternate forms based on gen cap
for form, gen in alternate_forms.items():
    if gen == 'remove' or gen > generation_cap:
        pokedex_df = pokedex_df[~pokedex_df['Pokémon'].str.contains(form, regex=True, case=False)]

# Modify pre gen 3 typings - research this
if generation_cap < 3:
    pokedex_df['Type'] = pokedex_df['Type'].str.replace(r'(^Type\||\|?Type)', '', regex=True).replace('','Type')

# Fairy type exists only after gen 5
if generation_cap < 6:
    pokedex_df['Type'] = pokedex_df['Type'].str.replace(r'(^Fairy\||\|?Fairy)', '', regex=True).replace('','Normal')
    pokedex_df.loc[pokedex_df['Pokémon'].isin(['TOGETIC', 'TOGEKISS']), 'Type'] = 'Normal|Flying'

# Assign gen based on pokedex 
gen_ranges = {
    1: range(1, 152),
    2: range(152, 252),
    3: range(251, 387),
    4: range(386, 494),
    5: range(493, 650),
    6: range(649, 722),
    7: range(721, 810),
    8: range(809, 906),
    9: range(905, 1025)
}
def assign_gen(input):
    for gen, range in gen_ranges.items():
        if input in range:
            return int(gen)

pokedex_df['Generation'] = pokedex_df['#'].apply(assign_gen)
pokedex_df = pokedex_df[pokedex_df['Generation'] <= generation_cap]

# Assign a Boolean to Pokemon that exist in the regional Kanto pokedex from the video game. Pokemon from a single generation may also exist in more than one regional pokedex. 
pokedex_df['Kanto Pokedex'] = np.where(pokedex_df['#'] <= 151, True, False)

### Apply Regional Pokedex Booleans to National Pokedex

In [ ]:
for region, link in regional_pokedex_urls.items():
    scrape_table(link)
    
    #Apply transformations
    df.columns = df.iloc[1]
    df = df.iloc[2:]
    df = df[['Pokémon']].drop_duplicates()
    df['Pokémon'] = df['Pokémon'].str.upper()
    df = df[df['Pokémon'] != 'Pokémon']
    df = df[df.index <= regional_index_cap[region]]
    df[f'{region} Pokedex'] = True
    pokedex_df = join(pokedex_df, df, 'left', 'Pokémon')
    
for col in pokedex_df.columns:
    pokedex_df[col] = pokedex_df[col].fillna(False).infer_objects(copy = False)

### Final National Pokedex Dataframe

##### Each regional pokedex column will specify the Pokemon that exist in that region's pokedex, as is in the video games.
##### Regional pokedexes may contain Pokemon from previous generations, a distinct attribute worth including separate from generation.

In [ ]:
pokedex_df